# Bước 02/06 — Chuẩn bị parent V6.2E (low disk)

V6.2E **bất biến** (17,483 / 2,272 / 2,012). Tải freeze + split manifest, đối chiếu config (lệch là dừng), stream ≤1200 ảnh TRAIN có insulator làm background, pHash VAL+TEST một lần, rồi xoá archive lớn. pHash cache được lưu lên Camber để Kaggle/local khác không phải tính lại.

| | Camber Stash |
|---|---|
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p2e/v6p2e_23562a45b343/v6p2e.freeze.json` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p2e/v6p2e_23562a45b343/v6p2e_split_manifest.csv` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p2e/v6p2e_23562a45b343/v6p2e_reconciled_labels.zip` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/curated_v6_rs1280_56b7ea194ee5.tar` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p1/v6p1_5d0b1149891c/v6p1_fotl_overlay_ee2d3c4497fd.tar.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/parent_cache/v6p2e_23562a45b343/parent_val_test_phash.csv.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/parent_cache/v6p2e_23562a45b343/parent_train_background_pool.csv` |
| **Output** | `(local) data/parent/train_background_pool/` |

Chạy lại notebook bất cứ lúc nào: bước đã xong + đúng config sẽ được **reuse**, bước dở dang sẽ **resume**.
Bước tiếp theo: **03_build_external_stage_v6p5**

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
# ===== DATASET TRÊN CAMBER — điền tag nếu muốn chọn tay, None = tự động =====
PARENT_DATASET = "v6p2e_23562a45b343"   # V6.2E: cố định, KHÔNG đổi
EXTERNAL_STAGE = None   # vd "v6p5_ext_0123456789ab"     (tạo ở bước 03)
COMPOSE        = None   # vd "v6p5_compose_0123456789ab" (tạo ở bước 04)
REQUIRE_CAMBER = True   # lưu chính trên Camber -> dừng nếu chưa đăng nhập

from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
from yolov11sdi.reporting import status
cfg = pipeline.load(stage_tag=EXTERNAL_STAGE, compose_tag=COMPOSE, parent_tag=PARENT_DATASET)
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
if REQUIRE_CAMBER and not cam.available():
    raise RuntimeError("Camber chưa sẵn sàng: chạy `camber login` hoặc export CAMBER_API_KEY "
                       "(CAMBER_BIN=... nếu CLI không nằm trong PATH). Chỉ chạy local: REQUIRE_CAMBER=False")
display(status.dataset_card(cfg, paths, cam))

In [ ]:
outcome = pipeline.run_stage("parent", cfg, force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
# Kiểm tra output đã thật sự nằm trên Camber chưa
display(pipeline.verify_on_camber("parent", cfg))

In [ ]:
pool = pd.read_csv(paths.resolve(store.load("parent").output("bg_pool_csv")["path"]))
for p in pool["image"].head(3):
    display(IPImage(filename=str(paths.resolve(p)), width=480))